# Practice Lab: Evaluation, Merging & Deployment (Lesson 9.6)

Module 9 · 8 exercises · Benchmarks + LLM-judge + safety + quantization + deploy

Exercises 1, 2, 3, 4 run locally (pure Python).


# Lesson 9.6: Evaluation, Merging & Deployment

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4 run **locally** (pure Python/numpy). Ex 5, 6, 7, 8 are design/architecture.


In [ ]:
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Benchmark Portfolio


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

benchmarks = [
    ("MMLU-Pro",   "Knowledge",     0.62, 0.03, "Active"),
    ("EvalPlus",   "Code",          0.71, 0.08, "Active"),
    ("MT-Bench",   "Conversation",  7.2,  0.8,  "Active"),   # /10 scale
    ("IFEval",     "Format",        0.68, 0.15, "Active"),
    ("MMLU",       "Knowledge",     0.86, 0.01, "SATURATED"),
    ("GSM8K",      "Math",          0.94, 0.01, "SATURATED"),
]

print("Benchmark Portfolio (Base vs Fine-Tuned):")
print("=" * 68)
print(f"  {'Benchmark':<12} {'Domain':<12} {'Base':>8} {'Tuned':>8} {'Delta':>8} {'Status'}")
print(f"  {'-'*62}")

active_deltas = []
for name, domain, base, boost, status in benchmarks:
    noise = np.random.normal(0, 0.01)
    tuned = base + boost + noise
    delta = tuned - base
    is_10 = name == "MT-Bench"

    if is_10:
        print(f"  {name:<12} {domain:<12} {base:>6.1f}/10 {tuned:>6.1f}/10 {delta:>+7.1f} {status}")
        if status == "Active": active_deltas.append(delta / 10)
    else:
        print(f"  {name:<12} {domain:<12} {base:>7.0%} {tuned:>7.0%} {delta:>+7.0%} {status}")
        if status == "Active": active_deltas.append(delta)

avg = np.mean(active_deltas) * 100
print(f"\nActive benchmarks avg improvement: +{avg:.1f}%")
print(f"IFEval largest gain = FT excels at format compliance")
print(f"MMLU/GSM8K saturated (86-96%+), uninformative for FT eval")
print(f"Always use ACTIVE benchmarks + compare base vs tuned")


</details>


---
## Exercise 2 (Easy): LLM-as-Judge with Bias Mitigation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

class LLMJudge:
    def __init__(self, position_bias=0.15):
        self.bias = position_bias

    def judge(self, resp_a, resp_b):
        q_a = len(resp_a.split()) / 50
        q_b = len(resp_b.split()) / 50
        diff = q_a - q_b + self.bias + np.random.normal(0, 0.1)
        if diff > 0.1: return "A"
        elif diff < -0.1: return "B"
        return "tie"

    def judge_with_swap(self, resp_a, resp_b):
        r1 = self.judge(resp_a, resp_b)
        r2 = self.judge(resp_b, resp_a)
        swap = {"A":"B","B":"A","tie":"tie"}
        r2m = swap[r2]
        if r1 == r2m: return r1, True
        return "tie", False

pairs = [
    ("Full refund within 7 days. 50% from 8-30 days. None after 30. Contact support.",
     "We have a refund policy. Check the website."),
    ("14999 rupees with lifetime access and GCP credits.",
     "It costs money. See pricing page."),
    ("Basic Python and high school math. No ML needed. We teach from scratch.",
     "You need some programming background."),
    ("Yes, EMI via Razorpay starting 2500 per month for 6 months.",
     "Payment options available. Contact us."),
    ("Daily 7 PM IST. Recordings within 2 hours. Discord community access.",
     "Evening classes available. Check schedule."),
]

judge = LLMJudge(position_bias=0.15)

print("LLM-as-Judge with Position Swap:")
print("=" * 55)

# Without swap
print(f"\n  WITHOUT position swapping:")
naive = {"A":0,"B":0,"tie":0}
for a, b in pairs:
    naive[judge.judge(a, b)] += 1
print(f"    A wins: {naive['A']}, B wins: {naive['B']}, ties: {naive['tie']}")
print(f"    A win rate: {naive['A']/len(pairs):.0%} (inflated by position bias!)")

# With swap
print(f"\n  WITH position swapping:")
swap_r = {"A":0,"B":0,"tie":0}
consistent = 0
for a, b in pairs:
    result, ok = judge.judge_with_swap(a, b)
    swap_r[result] += 1
    if ok: consistent += 1
print(f"    A wins: {swap_r['A']}, B wins: {swap_r['B']}, ties: {swap_r['tie']}")
print(f"    Consistent: {consistent}/{len(pairs)} ({consistent/len(pairs):.0%})")
print(f"    Bias-corrected A win rate: {swap_r['A']/len(pairs):.0%}")

print(f"\n  3 Biases: position (+15%), verbosity (50%->64%), self-enhancement")
print(f"  Mitigation: swap + ensemble (3 judges) + calibrate (30+ labeled)")
print(f"  Cost: LLM $0.01-0.10/judgment vs human $5/judgment")


</details>


---
## Exercise 3 (Medium): Safety Regression Test


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

categories = [
    "Harmful instructions",
    "Toxic content",
    "Bias (gender/race)",
    "Privacy violations",
    "Misinformation",
]
models = {
    "Base aligned":          ([0.98, 0.95, 0.92, 0.97, 0.90], 8.2),
    "FT (no safety)":        ([0.12, 0.15, 0.35, 0.08, 0.20], 7.5),
    "FT + 10% safety mix":   ([0.88, 0.85, 0.82, 0.90, 0.80], 7.8),
}

print("Safety Regression Test:")
print("=" * 68)
print(f"  {'Category':<24}", end="")
for m in models: print(f" {m:>14}", end="")
print()
print(f"  {'-'*68}")

for i, cat in enumerate(categories):
    print(f"  {cat:<24}", end="")
    for m_name, (rates, _) in models.items():
        r = np.clip(rates[i] + np.random.normal(0, 0.02), 0, 1)
        flag = " !!" if r < 0.5 else ""
        print(f" {r:>13.0%}{flag}", end="")
    print()

print(f"\n  Average Refusal Rate:")
for m_name, (rates, ppl) in models.items():
    avg = np.mean(rates)
    status = "PASS" if avg > 0.7 else "FAIL"
    print(f"    [{status}] {m_name:<26} {avg:.0%} (ppl: {ppl})")

print(f"\nPre-Deploy Checklist:")
for name, desc in [
    ("HarmBench","400+ behaviors, 7 categories"),
    ("ToxiGen","274K statements, 13 groups"),
    ("BBQ","58K bias questions"),
    ("WikiText-2 ppl","< base + 10%"),
    ("CCR","Track correction ratio in production")]:
    print(f"    [ ] {name}: {desc}")

print(f"\nMitigation: 5-10% safety data in training mix")
print(f"FT degrades refusal 98%->12%. Safety mix restores to 88%.")
print(f"NEVER deploy without safety benchmarks")


</details>


---
## Exercise 4 (Medium): Quantization Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def quantization_comparison(params_b=7):
    fp16_gb = params_b * 2

    methods = [
        ("FP16 (baseline)",     16, 1.00, 350,  "Reference, not for prod"),
        ("FP8 (H100+)",          8, 0.99, 465,  "H100/H200 production"),
        ("AWQ 4-bit + Marlin",   4, 0.96, 741,  "GPU serving (vLLM)"),
        ("GPTQ 4-bit",           4, 0.95, 712,  "GPU + multi-LoRA"),
        ("GGUF Q6_K",            6, 0.98,  45,  "Local (multilingual min)"),
        ("GGUF Q4_K_M",          4, 0.94,  35,  "Local (VRAM constrained)"),
    ]

    print(f"Quantization Comparison ({params_b}B model):")
    print("=" * 72)
    print(f"  {'Method':<24} {'Size':>8} {'Quality':>9} {'Tok/s':>8} {'Target'}")
    print(f"  {'-'*68}")

    for name, bits, quality, toks, target in methods:
        size = fp16_gb * bits / 16
        print(f"  {name:<24} {size:>6.1f}GB {quality:>8.0%} {toks:>7} {target}")

    print(f"\n  Size Reduction from FP16 ({fp16_gb}GB):")
    for name, bits, _, _, _ in methods:
        size = fp16_gb * bits / 16
        red = (1 - size / fp16_gb) * 100
        if red > 0:
            print(f"    {name:<24} -> {size:.1f}GB ({red:.0f}% smaller)")

    return methods

methods = quantization_comparison(7)

print(f"\nRecommendations:")
recs = [
    ("GPU prod (vLLM)",    "AWQ + Marlin",  "741 tok/s, 96% quality"),
    ("GPU + multi-LoRA",   "GPTQ 4-bit",    "Compatible with LoRA serving"),
    ("H100/H200",          "FP8",            "99% quality, 33% faster"),
    ("Local / Ollama",     "GGUF Q4_K_M",   "Fits consumer GPUs"),
    ("Multilingual local", "GGUF Q6_K",      "98% quality, preserves non-English"),
]
for target, method, why in recs:
    print(f"  {target:<22} -> {method}: {why}")

print(f"\nAWQ WITHOUT Marlin kernel is SLOWER than FP16!")
print(f"AutoAWQ archived May 2025 -> use llm-compressor")


</details>


---
## Exercise 5 (Medium): vLLM Production Config
Design/architecture exercise. See practice-lab-9_6.html for full solution.


In [ ]:
# Exercise 5: vLLM Production Config
pass


---
## Exercise 6 (Challenge): Canary Deployment Pipeline
Design/architecture exercise. See practice-lab-9_6.html for full solution.


In [ ]:
# Exercise 6: Canary Deployment Pipeline
pass


---
## Exercise 7 (Challenge): Production Monitoring Dashboard
Design/architecture exercise. See practice-lab-9_6.html for full solution.


In [ ]:
# Exercise 7: Production Monitoring Dashboard
pass


---
## Exercise 8 (Challenge): Full Production Checklist
Design/architecture exercise. See practice-lab-9_6.html for full solution.


In [ ]:
# Exercise 8: Full Production Checklist
pass
